# Census MTIS 已存档目录体积修复

在前版HTTP200但冻结体积上限过小之后，只读已NAS冷恢复的精确900648字节目录和三个原始PDF；零新GET，验证链接、制造业合计调整后库存销售比与发布时钟。Stage1 f696+b472 0bp费用后SOXX硬门不变。

In [1]:
import hashlib,io,json,os,pathlib,re,shlex,subprocess,sys,tarfile,tempfile,urllib.parse
from bs4 import BeautifulSoup
from pypdf import PdfReader
import qlib
from qlib.data import D
from qlib.workflow import R
root=pathlib.Path.cwd() if (pathlib.Path.cwd()/'src').exists() else pathlib.Path.cwd().parents[1];sys.path.insert(0,str(root))
from src.data.nas_market_data import archive_nas_recorder,_settings,_ssh
card=root/'research/stage1_census_mtis_saved_index_size_repair_20261023';p=json.loads((card/'plan.json').read_text());s=p['sources'];sha=lambda x:hashlib.sha256(pathlib.Path(x).read_bytes()).hexdigest();assert p['frozen'] and not (card/'run_state.json').exists();i_card=root/s['index_card'];pdf_card=root/s['PDF_card'];i_summary=json.loads((i_card/'summary.json').read_text());pdf_summary=json.loads((pdf_card/'summary.json').read_text())
assert sha(i_card/'summary.json')==s['index_summary_sha256'] and sha(i_card/'result.json')==s['index_result_sha256'];assert i_summary['source_NAS_sha256']==s['saved_index_NAS_archive_sha256'] and i_summary['source_NAS_cold_restore_verified'];assert pdf_summary['source_NAS_sha256']==s['saved_PDF_NAS_archive_sha256'] and pdf_summary['source_NAS_cold_restore_verified']
index=root/s['exact_saved_index_path'];assert index.stat().st_size==s['exact_saved_index_bytes'] and index.stat().st_size<=950000 and sha(index)==s['exact_saved_index_sha256'];state={'status':'saved_byte_repair_running','plan_sha256':sha(card/'plan.json'),'saved_index_reads':1,'saved_PDF_reads':0,'external_GETs':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
index_url='https://www.census.gov/mtis/historic_releases.html';soup=BeautifulSoup(index.read_bytes(),'html.parser');links={urllib.parse.urljoin(index_url,a.get('href')) for a in soup.find_all('a') if a.get('href')};months={'2011-07':'mtis1107.pdf','2018-07':'mtis1807.pdf','2025-07':'mtis2507.pdf'};exact={m:('https://www2.census.gov/mtis/historical/'+fn in links) for m,fn in months.items()}
qlib.init(provider_uri=str(root/'data/qlib/soxl'),region='us',kernels=1);calendar=[str(x)[:10] for x in D.calendar(start_time='2011-09-01',end_time='2025-10-31',freq='day')];mon=['JANUARY','FEBRUARY','MARCH','APRIL','MAY','JUNE','JULY','AUGUST','SEPTEMBER','OCTOBER','NOVEMBER','DECEMBER'];cases=[]
if all(exact.values()):
 for month in months:
  fp=root/'data/snapshots/stage1_census_mtis_manufacturer_ratio_source_pilot_20261023'/(month+'.response');assert sha(fp)==s['saved_PDF_SHA256'][month];state['saved_PDF_reads']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');data=fp.read_bytes();c={'economic_month':month,'PDF_sha256':sha(fp),'index_link_exact':True,'PDF_magic':data.startswith(b'%PDF'),'printed_release_date':None,'printed_release_time_ET':None,'first_Qlib_trade_after_release':None,'manufacturers_adjusted_current_preliminary_ratio':None,'manufacturers_adjusted_previous_revised_ratio':None,'status':'unknown'}
  if c['PDF_magic']:
   t='\n'.join(pg.extract_text() or '' for pg in PdfReader(io.BytesIO(data)).pages);head=t[:1600].upper();date=re.search(r'(JANUARY|FEBRUARY|MARCH|APRIL|MAY|JUNE|JULY|AUGUST|SEPTEMBER|OCTOBER|NOVEMBER|DECEMBER)\s+(\d{1,2}),\s+(20\d{2})',head);time_ok=bool(re.search(r'10:00\s*(?:A\.?M\.?)?\s*E(?:D|S)T',head))
   if date and time_ok:
    release=f'{int(date.group(3)):04d}-{mon.index(date.group(1))+1:02d}-{int(date.group(2)):02d}';c['printed_release_date']=release;c['printed_release_time_ET']='10:00';c['first_Qlib_trade_after_release']=next((d for d in calendar if d>release),None)
   start=re.search(r'(?im)^\s*Adjusted\s*\d*\s*$',t);adjusted=t[start.end():].split('Not Adjusted',1)[0] if start else '';matches=list(re.finditer(r'Manufacturers(?:²|3|2)?[^\d]{0,100}?((?:\d[\d,]*\s+){6})(\d+\.\d\d)\s+(\d+\.\d\d)\s+(\d+\.\d\d)',adjusted,re.I|re.S))
   if len(matches)==1:
    mm=matches[0];c['manufacturers_adjusted_current_preliminary_ratio']=float(mm.group(2));c['manufacturers_adjusted_previous_revised_ratio']=float(mm.group(3));c['same_release_prior_year_ratio']=float(mm.group(4))
   else:c['parse_diagnostic']={'adjusted_row_matches':len(matches),'adjusted_chars':len(adjusted),'time_pattern':time_ok,'date_pattern':bool(date)}
   if c['printed_release_date'] and c['first_Qlib_trade_after_release'] and c['manufacturers_adjusted_current_preliminary_ratio'] is not None:c['status']='original_sample_semantics_and_clock_qualified'
  cases.append(c)
status='three_saved_original_MTiS_samples_semantics_and_clock_qualified' if len(cases)==3 and all(c['status']=='original_sample_semantics_and_clock_qualified' for c in cases) else 'saved_original_MTiS_source_layout_or_clock_unknown';result={'status':status,'plan_sha256':sha(card/'plan.json'),'saved_index_sha256':sha(index),'exact_links':exact,'cases':cases,'saved_index_reads':state['saved_index_reads'],'saved_PDF_reads':state['saved_PDF_reads'],'external_GETs':0,'2011_first_public_byte_vintage_qualified':False,'full_2011_2026_panel_qualified':False,'semiconductor_specific':False,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
with R.start(experiment_name=p['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=p,index_source_NAS=s['saved_index_NAS_archive_sha256'],PDF_source_NAS=s['saved_PDF_NAS_archive_sha256']);rec.save_objects(artifact_path='analysis',cases=cases,result=result);rec.log_metrics(qualified_samples=sum(c['status']=='original_sample_semantics_and_clock_qualified' for c in cases),source_reads=state['saved_index_reads']+state['saved_PDF_reads'],external_GETs=0,model_fits=0,native_accounts=0);rid,eid=rec.id,str(rec.experiment_id)
state.update(status='finished',recorder_id=rid,experiment_id=eid);(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');ev=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);assert _ssh(_settings(root)['ssh'],'sha256sum '+shlex.quote(ev['nas_path'])).split()[0]==ev['sha256']
summary={'experiment':p['experiment'],'decision':status,'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_index_NAS_sha256':s['saved_index_NAS_archive_sha256'],'source_PDF_NAS_sha256':s['saved_PDF_NAS_archive_sha256'],'sources_cold_restored':True,'recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':ev,'NAS_readback':{'sha256':ev['sha256'],'remote_hash_verified':True},'qualified_samples':sum(c['status']=='original_sample_semantics_and_clock_qualified' for c in cases),'external_GETs':0,'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*']},'limitations':['Current July2011 PDF Last-Modified in2014 means first-public 2011 byte vintage remains unqualified.','Manufacturers aggregate does not isolate semiconductor demand; three samples are not a full panel.','No Stage1 factor, predictive gate, funded 0bp account or SOXX success.']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');print(json.dumps({'decision':status,'links':exact,'qualified':summary['qualified_samples'],'recorder_id':rid}))


2026/10/23 14:48:15 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[11042:MainThread](2026-10-23 14:48:17,049) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[11042:MainThread](2026-10-23 14:48:17,051) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[11042:MainThread](2026-10-23 14:48:17,051) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[11042:MainThread](2026-10-23 14:48:17,247) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_census_mtis_saved_index_size_repair_20261023.


[11042:MainThread](2026-10-23 14:48:17,249) INFO - qlib.workflow - [exp.py:258] - Experiment 152094778758056102 starts running ...


[11042:MainThread](2026-10-23 14:48:17,329) INFO - qlib.workflow - [recorder.py:345] - Recorder 71bae66c94f9495aaae2dc01bb2c070e starts running under Experiment 152094778758056102 ...


[11042:MainThread](2026-10-23 14:48:18,430) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{"decision": "three_saved_original_MTiS_samples_semantics_and_clock_qualified", "links": {"2011-07": true, "2018-07": true, "2025-07": true}, "qualified": 3, "recorder_id": "71bae66c94f9495aaae2dc01bb2c070e"}


# 结论：Census MTIS 制造业合计库存销售比原始来源样本

完整 Stage1 f696+b472 仍是最终经济父方案；本卡只做来源与时钟核验。先前固定四次 GET 中，三个 Census PDF 已收到，但目录 HTTP500；单次目录重取虽为HTTP200，其900,648字节超过旧700,000字节上限。两次失败收据均保留。本次零新网络请求，仅使用 NAS 已逐件冷恢复的精确目录和三份PDF，Qlib Recorder `71bae66c94f9495aaae2dc01bb2c070e` 的选定工件也已冷读回。

目录精确链接 **3/3**；2011、2018、2025 年7月各有唯一的 Table1 **Adjusted Manufacturers** 库存销售比。原稿当期初值/上月同版修订值分别为 **1.32/1.33、1.35/1.34、1.56/1.57**。三份报告都印有美国东部时间10:00的发布日期；严格晚于该日期的首个 Qlib 美国交易日分别是 **2011-09-15、2018-09-17、2025-09-17**。同日09:46的 Stage1决策不能使用新公布值。

**决定：三年份来源结构与保守发布时钟可行，未晋级因子。** 这项是全美制造业合计，并非半导体订单或 ISM 客户库存。2011 当前 PDF 的服务器修改时间为2014年，不能证明当年的首次公开字节；2018与2025的服务器时间也不能单独当作不可变初版。原稿比率经季调、未扣价格变动，且当期/上期属于初值/修订值。完整2011–2026原始逐月版本、缺失、修订、信息门及完整0bp费用后SOXX账户均未核实；此轮0收益读取、0拟合、0账户。
